# A proposal is not a decision

A model can write an answer and attach a high confidence score. That is a *proposal*. Standing is something else: the pipeline actually produced a terminal, and seven halt guards derived from evidence all hold.

This notebook uses the shipped calculator. Arithmetic is exact (fractions, not floating point).

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "coda_fda_core.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "fda" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "fda"
elif (p / "fda" / "code").is_dir():
    root = p / "fda"
else:
    root = p
assert (root / "code").is_dir(), "run from the package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from sympy import Rational
from coda_fda_core import DescentPacket, GuardEvidence, evaluate_authorization

## A five-field packet, in words

The packet is five pieces of finite data:

- $K$ — where you are (the carrier)
- $E$ — which candidates may be tried there
- $V$ — how they are checked
- $\tau$ — how a check becomes a terminal
- $\Lambda$ — what happens on failure (refuse, lift, or roll back; do not silently promote)

Without those five, standing is empty. Confidence is not one of the five.

In [2]:
packet = DescentPacket(
    carrier_k="toy_carrier",
    evaluator_class_e=["candidate_a", "candidate_b"],
    verifier_v="replay_v1",
    terminalizer_tau="tau_standing",
    discipline_lambda="no_self_promote",
)
assert packet.is_valid()
print("packet accepted")

packet accepted


## Standing, when the pipeline really ran

Evidence is a finite record, not a mood. The seven guards are *derived* from it. If the packet is valid and every derived guard holds, the calculator reports standing inhabited.

In [3]:
green = GuardEvidence(
    pipeline_accepted=True,
    replay_deterministic=True,
    residual_mass=Rational(0, 1),
    residual_zero_tol=Rational(0, 1),
    skeleton_preserved=True,
    candidates_checked=2,
    candidates_declared=2,
    checked_candidate_ids=("candidate_a", "candidate_b"),
    pending_lifts=0,
    active_refusal=False,
)
stood = evaluate_authorization("proposal_ok", packet, green)
print("inhabited:", stood.inhabited)
print("terminal:", stood.terminal_state)
print("refusal:", stood.refusal_code)
assert stood.inhabited is True
assert stood.refusal_code is None

inhabited: True
terminal: term_standing_tau_standing_proposal_ok
refusal: None


## The same proposal, with no packet

Keep the green evidence. Drop the packet. Attach a confidence of $999/1000$. Standing is still empty: there is no five-field packet, so there is nothing for the pipeline to have produced.

In [4]:
fluent = evaluate_authorization(
    "proposal_fluent",
    None,
    green,
    logit_confidence=Rational(999, 1000),
)
print("inhabited:", fluent.inhabited)
print("refusal:", fluent.refusal_code)
print("confidence was supplied and ignored")
assert fluent.inhabited is False
assert fluent.refusal_code == "R_nopacket"

inhabited: False
refusal: R_nopacket
confidence was supplied and ignored
